In [1]:
from ETGEMs_function_PMI import *
import pandas as pd
import cobra
import ast
from cobra.io import write_sbml_model
from numpy import *
import copy
import math

In [2]:
#Get Reaction G0 from local file_original
reaction_g0_file_original0 = './reaction_g0_strain0.txt'
reaction_g0_file_original1 = './reaction_g0_strain1.txt'
#Get Metabolite concentration from local file
metabolites_lnC_file_original0 = './metabolites_lnC_strain0.txt'
metabolites_lnC_file_original1 = './metabolites_lnC_strain1.txt'
#Get Model from local file
model_file_original0 = './iML1515.xml'
model_file_original1 = './iMM904.xml'
#Get reaction kcat data from ECMpy
reaction_kcat_MW_file_original0 = './ID_kcat_MW_file_strain0.csv'
reaction_kcat_MW_file_original1 = './ID_kcat_MW_file_strain1.csv'

In [3]:
## Convert to usable model formats
model0=Get_Concretemodel_Need_Data(reaction_g0_file_original0,metabolites_lnC_file_original0,model_file_original0,reaction_kcat_MW_file_original0)
model1=Get_Concretemodel_Need_Data(reaction_g0_file_original1,metabolites_lnC_file_original1,model_file_original1,reaction_kcat_MW_file_original1)

In [4]:
# get the information of km, vmax and public metabolites
km = pd.read_csv('./km.csv')
vmax = pd.read_csv('./vmax.csv')
public_metabolism = pd.read_csv('./public_metabolism_test.csv')

'''
public_metabolism_name_list = []
public_metabolism_concentration_list = []
for i in public_metabolism['metabolism']:
    public_metabolism_name_list.append(i)
for j in public_metabolism['concentration']:
    public_metabolism_concentration_list.append(j)
public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
print(public_metabolism)
'''

"\npublic_metabolism_name_list = []\npublic_metabolism_concentration_list = []\nfor i in public_metabolism['metabolism']:\n    public_metabolism_name_list.append(i)\nfor j in public_metabolism['concentration']:\n    public_metabolism_concentration_list.append(j)\npublic_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))\nprint(public_metabolism)\n"

In [5]:
# definate the function to simulate metabolism and interaction
def time_space_state(model_list, biomass_list, growth_list, breed_list, parameter_list, public_metabolism, vmax, km, deta_t, deta_s, micro_distribute_threshold, length, D):
    
    number_cell_side = length // deta_s
    number_cell_side = int(number_cell_side)
    public_metabolism_name_list = []
    public_metabolism_concentration_list = []
    for i in public_metabolism['metabolism']:
        public_metabolism_name_list.append(i)
    for j in public_metabolism['concentration']:
        public_metabolism_concentration_list.append(j)
    public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
    
    number_model = len(model_list)
    
    k_m = {}
    v_max = {}
    for i in range(number_model):
        for j in range(len(km['reactions_strain'+str(i)])):
            k_m[(i, km['reactions_strain'+str(i)][j])] = km['km_strain'+str(i)][j]
            v_max[(i, vmax['reactions_strain'+str(i)][j])] = vmax['vmax_strain'+str(i)][j]
    
    
    number_public_metabolism = len(public_metabolism)
    distribute_micro_list = {}
    distribute_public_metabolism_list = {}
    distribute_lb_list = {}
    public_metabolism_r_list = []
    #set the initial distribution of straints
    for i in range(number_model):
        distribute_micro = np.random.randint(40, size=number_cell_side)
        distribute_micro_list.update({i: distribute_micro})
    print(distribute_micro_list)
    #set the initial distribution of metabolites
    for i in public_metabolism_name_list:
        distribute_public_metabolism = multiply(np.mat(np.ones(number_cell_side)), public_metabolism[i])
        distribute_public_metabolism_list.update({i: distribute_public_metabolism})
    #calculate the upperbounds of uptake reactions for public metabolites
    public_reaction_ub_list = {}
    public_reaction_list = {}
    for i in range(number_model):
        public_reaction_ub = {}
        public_reaction = []
        for rea in model_list[i]['reaction_list']:
            if 'EX_' in rea:
                for j in public_metabolism_name_list:
                    try:
                        model_list[i]['coef_matrix'][(j,rea)]
                    except:
                        pass
                    else:
                        ub = np.mat(np.ones(number_cell_side))
                        if model_list[i]['coef_matrix'][(j,rea)] > 0:
                            for m in range(number_cell_side):
                                ub[0,m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                        else:
                            try:
                                model_list[i]['ub_list'][rea]
                            except:
                                ub = ub * 1000
                            else:
                                ub = ub * model_list[i]['ub_list'][rea]
                        public_reaction_ub.update({rea: ub})
                        public_reaction.append(rea)
                        break
            public_reaction_ub_list[i] = public_reaction_ub
            public_reaction_list[i] = public_reaction

            
    ct = 0
    
    distribute_micro_list_t = {ct: distribute_micro_list}
    distribute_public_metabolism_list_t = {ct: distribute_public_metabolism_list}
    distribute_lb_list_t = {ct: distribute_lb_list}
    r = micro_distribute_threshold + 1
    r_threshold = [r]*5
    slip_r = np.mean(r_threshold[-5:])
    number = {}
    various = {}
    for i in range(number_model):
        number[i] = [np.mean(distribute_micro_list[i])]
        various[i] = [np.std(distribute_micro_list[i])/np.mean(distribute_micro_list[i])]
    
            
    # iterative simulation by slip_r
    while slip_r >= micro_distribute_threshold:
        ct += 1
        print(ct)
        distribute_micro_list_t[ct] = copy.deepcopy(distribute_micro_list_t[ct-1])
        micro_decrease = {}
        micro_increase = {}
        
        #simulate the cell wandering
        #micro_decrease: the decrease number of strains
        #micro_increase: the increase number of strains
        #micro_decrease_cell: the decrease number of strains in the current grid
        #micro_increase_fcell: the increase number of strains in the forward grid
        #micro_increase_bcell: the increase number of strains in the backward grid
        for m in range(number_cell_side):
            micro_decrease_cell = {}
            micro_increase_cell = {}
            for i in range(number_model):
                micro_decrease_cell[i] = 0
                micro_increase_cell[i] = 0
            micro_decrease[m] = micro_decrease_cell
            micro_increase[m] = micro_increase_cell
            
            
        if ct > 0:
            met = 'glc__D_e'
            threshold = 0.3
            for m in range(number_cell_side):
                #calculate the number of strains in the internal grids
                if 0<m<number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = (distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * micro_decrease_cell
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_increase_bcell = micro_decrease_cell-micro_increase_fcell
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                                    micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                                else:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                            elif distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the first grid
                elif m == 0:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the last grid
                elif m == number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_fcell = int(micro_increase_fcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
        
            #calculate the number of strains after wandering
            for m in range(number_cell_side):
                for i in range(number_model):
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] - micro_decrease[m][i]
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] + micro_increase[m][i]
                    distribute_micro_list_t[ct][i][m] = max(0, distribute_micro_list_t[ct][i][m])
                            
                            
        
        #simulate the substrate diffusion by Fick's second law
        for m in range(number_cell_side): 
            if 0<m<number_cell_side-1:
                for met in public_metabolism_name_list:
                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + ((distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s-(distribute_public_metabolism_list[met][0,m]-distribute_public_metabolism_list[met][0,m-1])/deta_s)/deta_s*D*deta_t
                    if distribute_public_metabolism_list[met][0,m] < 0:
                        print('Warning: the D is too high!')
                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == 0:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == number_cell_side-1:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m-1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            
        #simulate the metabolism by ETMs
        biomass_value_list = {}
        number_model_range = []
        for m in range(number_cell_side):
            if m%2 == 0:
                number_model_range.append([0,1])
            elif m%2 == 1:
                number_model_range.append([1,0])
        for m in range(number_cell_side):
            B_value_list = []
            
            
            biomass_value_micro = {}
            for i in number_model_range[m]:
                if distribute_micro_list_t[ct][i][m] > 0:
                    public_metabolism_flux_list = {}
                    for j in public_metabolism_name_list:
                        public_metabolism_flux_list.update({j: 0})
                    for j in public_reaction_list[i]:
                        model_list[i]['ub_list'][j] = int(public_reaction_ub_list[i][j][0,m])
            
                    biomass_id = biomass_list[i]
                    E_total=parameter_list[i][0] 
                    #set the carbon source to glucose
                    substrate_name='EX_glc__D_e_reverse'
                    substrate_value=parameter_list[i][1]
                    biomass_value=growth_list[i]
                    K_value=parameter_list[i][2]

                    try:
                        MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                    except:
                        save_rate = 0.9
                        while save_rate >= 0:
                            biomass_value2 = save_rate * biomass_value
                            try:
                                MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                            except:
                                save_rate = save_rate - 0.1
                            else:
                                #calculate the MDF values of metabolic networks
                                biomass_value_micro[i] = biomass_value2
                                B_value2 = MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                                #calculate the biomass yield under the MDF value
                                obj_name=biomass_list[i]
                                obj_target='maximize'
                                if i == 0:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                elif i == 1:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                biomass_value=max_biomass_under_mdf*0.9
                                
                                #calculate the minimum value of the sum of the fluxes
                                if i == 0:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                elif i == 1:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                
                                #translating the results of ETMs into a usable form
                                model=model_list[i]['model']
                                reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                                reaction_g0=model_list[i]['reaction_g0']
                                coef_matrix=model_list[i]['coef_matrix']
                                metabolite_list=model_list[i]['metabolite_list']
                                use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
                                distribute_public_metabolism_increase = {}
                                distribute_public_metabolism_decrease = {}
                                
                                #simulate the fluxes of the public metabolites
                                for rea in public_reaction_list[i]:
                                    for met in public_metabolism_name_list:
                                        try:
                                            model_list[i]['coef_matrix'][(met,rea)]
                                        except:
                                            pass
                                        else:
                                            public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
                                            
                                #simulate the distribution of the public metabolites
                                for met in public_metabolism_name_list:
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]*save_rate
                                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
                                    
                                    
                                save_rate_ub = [save_rate]
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        met = rea[3::]
                                        if met in public_metabolism_name_list:
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            save_rate_met = distribute_public_metabolism_list[met][0,m]/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                            save_rate_ub.append(save_rate_met)
                                        
                                save_rate_final = min(save_rate_ub)
                                
                                
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        if met in public_metabolism_name_list:
                                            met = rea[3::]
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - intracellular_c_dict[met]*int((save_rate_final)*distribute_micro_list_t[ct][i][m]) + intracellular_c_dict[met]*int((1-save_rate_final)*distribute_micro_list_t[ct][i][m])
                                            if distribute_public_metabolism_list[met][0,m] < 0:
                                                print(met + '_3: ', distribute_public_metabolism_list[met][0,m])
                                    
                                
                                distribute_micro_decrease = int((1-save_rate_final) * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_increase = int(save_rate_final * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] - distribute_micro_decrease + distribute_micro_increase)
                                
                                break
                            
                        continue
                    else:
                        #calculate the MDF values of metabolic networks
                        biomass_value_micro[i] = biomass_value
                        B_value=MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                        B_value_list.append(B_value)
                        #calculate the biomass yield under the MDF value
                        obj_name=biomass_list[i]
                        obj_target='maximize'
                        if i == 0:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        biomass_value=max_biomass_under_mdf*0.9
                        
                        #calculate the minimum value of the sum of the fluxes
                        if i == 0:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        
                        #translating the results of ETMs into a usable form
                        model=model_list[i]['model']
                        reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                        reaction_g0=model_list[i]['reaction_g0']
                        coef_matrix=model_list[i]['coef_matrix']
                        metabolite_list=model_list[i]['metabolite_list']
                        use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
            
                        #simulate the fluxes of the public metabolites
                        for rea in public_reaction_list[i]:
                            for met in public_metabolism_name_list:
                                try:
                                    model_list[i]['coef_matrix'][(met,rea)]  
                                except:
                                    pass
                                else:
                                    public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
            
                        #simulate the distribution of the public metabolites
                        distribute_public_metabolism_ori = copy.deepcopy(distribute_public_metabolism_list)
                        distribute_public_metabolism_re = copy.deepcopy(distribute_public_metabolism_ori)
                        for met in public_metabolism_name_list:
                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]
                            distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                            if distribute_public_metabolism_list[met][0,m] < 0:
                                print(met+'_1: ', distribute_public_metabolism_list[met][0,m])       
                        
                        #simulate the multiplication and death rates of strains and the distribution of public metabolites after multiplication or death
                        death_rate = 0
                        birth_rate = 1
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    bd_rate_met = (distribute_public_metabolism_list[met][0,m] - 0.1)/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                    if bd_rate_met < 0:
                                        death_rate_lb = (public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] - distribute_public_metabolism_re[met][0,m] + 0.1)/(public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] + intracellular_c_dict[met] * distribute_micro_list_t[ct][i][m])
                                        death_rate = max(death_rate, death_rate_lb)
                                    else:
                                        birth_rate = min(birth_rate, bd_rate_met)
                        death_rate = min(death_rate, 1)
                        birth_rate = min(birth_rate,1)
                        if death_rate > 0:
                            birth_rate = 0
                            for met in public_metabolism_name_list:
                                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + public_metabolism_flux_list[met] * deta_t * math.ceil(distribute_micro_list_t[ct][i][m] * death_rate)
                                distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                    
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_ori[met][0,m] - intracellular_c_dict[met]*math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i]) + intracellular_c_dict[met]*math.ceil(death_rate*distribute_micro_list_t[ct][i][m])
                                    if distribute_public_metabolism_list[met][0,m] < 0:
                                        print(met+'_2: ', distribute_public_metabolism_list[met][0,m])
                                        distribute_public_metabolism_list[met][0,m] = 0
                                
                        #simulate the distribution of strains after multiplication or death
                        distribute_micro_increase = math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i])
                        distribute_micro_decrease = math.ceil(distribute_micro_list_t[ct][i][m]*death_rate)
                        distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] + distribute_micro_increase - distribute_micro_decrease)
                
            #calculate the upperbounds of nutrient uptake rates after substrate diffusion, cell wandering, metabolism, multiplication and death
            biomass_value_list[m] = biomass_value_micro
            
            for i in range(number_model):
                for rea in model_list[i]['reaction_list']:
                    if 'EX_' in rea:
                        for j in public_metabolism_name_list:
                            try:
                                model_list[i]['coef_matrix'][(j,rea)]
                            except:
                                pass
                            else:
                                if model_list[i]['coef_matrix'][(j,rea)] > 0:
                                    public_reaction_ub_list[i][rea][0, m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                                    if i == 0:
                                        # gene upregulation for Fe3+ intake
                                        if 'EX_fe3_e' in rea:
                                            public_reaction_ub_list[i][rea][0, m] = public_reaction_ub_list[i][rea][0, m] * 1.2

        # calculate the mean number and uniformity of distribution of strains at this iteration
        for i in range(number_model):
            strain_number = np.mean(distribute_micro_list_t[ct][i])
            strain_various = np.std(distribute_micro_list_t[ct][i])
            number[i].append(strain_number)
            various[i].append(strain_various)
            print('strain_number: ', i, strain_number)
            print('strain_various: ', i, strain_various)
                                    
        # calculate the slip_r at this iteration
        if ct > 1:
            r = 0
            for i in range(number_model):
                for m in range(number_cell_side):
                    if distribute_micro_list_t[ct-1][i][m] > 0:
                        r = r + ((distribute_micro_list_t[ct][i][m]-distribute_micro_list_t[ct-1][i][m])/(distribute_micro_list_t[ct-1][i][m]))**2
                    else:
                        r = r + (distribute_micro_list_t[ct][i][m])**2
       
        r_threshold.append(r)
        slip_r = np.mean(r_threshold[-5:])
        fd_r_threshold = r_threshold[5:]
        print('fd_r_threshold: ', fd_r_threshold)
        print('slip_r: ', slip_r)
    return(distribute_micro_list_t, number, various)

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 1, 30, 29,  9, 17, 17, 16, 25,  2, 35, 12,  4, 22, 29,  5, 30,  7,
       29, 15]), 1: array([ 9, 35,  4, 34, 24, 19, 22, 15,  9, 18, 35, 38, 22, 16, 28,  5, 28,
       23,  7])}
1
strain_number:  0 20.736842105263158
strain_various:  0 12.838437785203578
strain_number:  1 20.789473684210527
strain_various:  1 10.724192755135814
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 24.57894736842105
strain_various:  0 6.531831285680789
strain_number:  1 20.894736842105264
strain_various:  1 6.919801768092443
fd_r_threshold:  [1.05, 312.0714843714069]
slip_r:  63.25429687428137
3
strain_number:  0 29.105263157894736
strain_various:  0 5.014659949032649
strain_number:  1 20.94736842105263
strain_various:  1 4.904065812756277
fd_r_threshold:  [1.05, 312.0714843714069, 8.180434473267724]
slip_r:  64.68038376893492
4
strain_number:  0 34.526315789473685
strain_various:  0 7.714432278136948
strain_number:  1 21.0
strain_various:  1 4.47213595499958
fd_r_threshold:  [1.05, 312

glc__D_e_1:  -10.061367471713591
glc__D_e_1:  -6.305560325377991
strain_number:  0 14.736842105263158
strain_various:  0 5.589860230463897
strain_number:  1 8.789473684210526
strain_various:  1 2.7640349485265014
fd_r_threshold:  [1.05, 312.0714843714069, 8.180434473267724, 6.202019397630516, 6.539131080820197, 6.972933096764219, 6.463370031542223, 4.817544164375834, 3.4755525327687966, 4.426874257212955, 4.184023134464104, 4.428136351152849]
slip_r:  4.266426087994908
13
glc__D_e_1:  -8.180601649288626
glc__D_e_1:  -4.579238924753819
glc__D_e_1:  -1.8290607657156674
glc__D_e_1:  -9.259513696597395
glc__D_e_1:  -8.178402743991837
glc__D_e_1:  -0.7228158870213957
glc__D_e_1:  -2.253785196787993
glc__D_e_1:  -10.631717158317697
glc__D_e_1:  -8.058813687046325
glc__D_e_1:  -1.7645278800319288
glc__D_e_1:  -8.048146702813455
glc__D_e_1:  -6.45280612357618
glc__D_e_1:  -1.076242547374227
glc__D_e_1:  -1.6236268198091937
glc__D_e_1:  -8.356591338716566
glc__D_e_1:  -8.098598856344314
glc__D_

glc__D_e_1:  -1.0560384581932798
glc__D_e_1:  -0.3947883124899776
glc__D_e_1:  -0.9090179391216942
glc__D_e_1:  -0.7240807783527048
glc__D_e_1:  -1.7848293904407528
glc__D_e_1:  -1.0068630921550246
glc__D_e_1:  -0.1059644481409674
glc__D_e_1:  -1.309274524862895
glc__D_e_1:  -0.5131321925869361
glc__D_e_1:  -1.0273618192186529
glc__D_e_1:  -0.7619325020936119
glc__D_e_1:  -1.6804619402513352
glc__D_e_1:  -0.6225333946122698
glc__D_e_1:  -1.2762273581554764
glc__D_e_1:  -1.2028302096098087
glc__D_e_1:  -1.7170598362415253
glc__D_e_1:  -0.18675645067129176
glc__D_e_1:  -0.7832312678892459
glc__D_e_1:  -0.5720797335131924
glc__D_e_1:  -1.8201828418345056
glc__D_e_1:  -0.842619949800407
strain_number:  0 2.473684210526316
strain_various:  0 0.8187552203212656
strain_number:  1 1.6842105263157894
strain_various:  1 0.5668594533825794
fd_r_threshold:  [1.05, 312.0714843714069, 8.180434473267724, 6.202019397630516, 6.539131080820197, 6.972933096764219, 6.463370031542223, 4.817544164375834, 3.

strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 312.0714843714069, 8.180434473267724, 6.202019397630516, 6.539131080820197, 6.972933096764219, 6.463370031542223, 4.817544164375834, 3.4755525327687966, 4.426874257212955, 4.184023134464104, 4.428136351152849, 5.127964814593813, 4.8528049599988075, 4.47496679196625, 3.247318594104309, 5.0791666666666675, 4.449965986394558, 5.345555555555556, 6.111111111111111, 5.611111111111111, 4.25, 5.861111111111111, 2.0, 5.0, 5.0, 1.25, 0.0]
slip_r:  2.65
29
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 312.0714843714069, 8.180434473267724, 6.202019397630516, 6.539131080820197, 6.972933096764219, 6.463370031542223, 4.817544164375834, 3.4755525327687966, 4.426874257212955, 4.184023134464104, 4.428

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([18, 33, 27, 18, 29, 29,  0, 38, 17, 17, 35, 11, 10, 35, 36, 17, 11,
       16, 25]), 1: array([27, 10, 37, 30, 36, 17, 38,  0, 26,  9, 18, 31,  6, 14, 31, 32, 18,
       20, 36])}
1
strain_number:  0 26.31578947368421
strain_various:  0 12.40989686364093
strain_number:  1 23.36842105263158
strain_various:  1 11.653824880657442
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 31.263157894736842
strain_various:  0 7.683489511955646
strain_number:  1 23.473684210526315
strain_various:  1 6.124742065912753
fd_r_threshold:  [1.05, 452.8635656695924]
slip_r:  91.41271313391847
3
strain_number:  0 37.21052631578947
strain_various:  0 7.309539170213566
strain_number:  1 23.57894736842105
strain_various:  1 5.6783595617941796
fd_r_threshold:  [1.05, 452.8635656695924, 9.514747452019906]
slip_r:  93.10566262432245
4
strain_number:  0 44.31578947368421
strain_various:  0 9.59195184340523
strain_number:  1 23.789473684210527
strain_various:  1 6.1007242656177265
fd_r_threshold:

glc__D_e_1:  -7.940969498359927
glc__D_e_1:  -6.345628919122652
glc__D_e_1:  -4.449172735617785
glc__D_e_1:  -2.1013936498270116
glc__D_e_1:  -4.069616282489484
glc__D_e_1:  -4.274957065519613
glc__D_e_1:  -4.591533484725578
glc__D_e_1:  -3.7191319549322497
glc__D_e_1:  -11.096011630029757
glc__D_e_1:  -8.523108158758383
glc__D_e_1:  -2.1055764632025906
glc__D_e_1:  -2.161168216971742
glc__D_e_1:  -6.632424496303779
glc__D_e_1:  -5.860202387299809
glc__D_e_1:  -3.824941529038682
glc__D_e_1:  -5.41150259257443
glc__D_e_1:  -11.938192336982418
glc__D_e_1:  -7.204822298612719
glc__D_e_1:  -2.3321994233879906
glc__D_e_1:  -5.830338807817849
glc__D_e_1:  -11.119599218274924
glc__D_e_1:  -7.20934765013853
glc__D_e_1:  -1.758082182000508
glc__D_e_1:  -2.3054664544354746
glc__D_e_1:  -4.594919069279392
glc__D_e_1:  -2.6397932852111947
glc__D_e_1:  -7.1663570715697
glc__D_e_1:  -6.3495472955455226
glc__D_e_1:  -6.846595039385479
glc__D_e_1:  -8.749069124111552
glc__D_e_1:  -4.624055131610652
gl

glc__D_e_1:  -1.5119641016121745
glc__D_e_1:  -0.6246576648895505
glc__D_e_1:  -1.8075613399537778
glc__D_e_1:  -0.7068449243272639
glc__D_e_1:  -0.9571628348250727
glc__D_e_1:  -1.4713924614567895
glc__D_e_1:  -0.9679632847299484
glc__D_e_1:  -1.5030312834979025
glc__D_e_1:  -1.1941424398963145
strain_number:  0 2.4210526315789473
strain_various:  0 0.6740130776245103
strain_number:  1 1.8421052631578947
strain_various:  1 0.8119604537127112
fd_r_threshold:  [1.05, 452.8635656695924, 9.514747452019906, 8.19041245537767, 7.54007316875312, 6.023096944284936, 3.60757693617369, 3.3105195158778296, 4.248223984586988, 4.234529491038918, 5.2136350535066915, 4.708269344986164, 4.283656308080911, 4.255632965919762, 4.50051085218293, 3.411111111111112, 5.299444444444444]
slip_r:  4.3500711363478315
18
glc__D_e_1:  -0.4600132663784886
glc__D_e_1:  -0.777684279379043
glc__D_e_1:  -1.2743975278152786
glc__D_e_1:  -0.34183796092750196
glc__D_e_1:  -0.792845800703684
glc__D_e_1:  -0.2593154265399395

strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 452.8635656695924, 9.514747452019906, 8.19041245537767, 7.54007316875312, 6.023096944284936, 3.60757693617369, 3.3105195158778296, 4.248223984586988, 4.234529491038918, 5.2136350535066915, 4.708269344986164, 4.283656308080911, 4.255632965919762, 4.50051085218293, 3.411111111111112, 5.299444444444444, 5.249999999999998, 4.083333333333334, 8.36111111111111, 3.111111111111111, 5.25, 4.5, 0.0, 4.5, 4.0, 1.0, 1.0]
slip_r:  2.1
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 452.8635656695924, 9.514747452019906, 8.19041245537767, 7.54007316875312, 6.023096944284936, 3.60757693617369, 3.3105195158778296, 4.248223984586988, 4.234529491038918, 5.2136350535066915, 4.708269344986164, 4.283656308080911, 4.255632965919762, 4.50051085218293, 3.41111111111111

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 5, 20, 28, 14,  5,  7,  3,  4, 15, 11, 27,  1, 34, 36,  1,  3, 35,
       29, 30]), 1: array([11, 10,  3,  5, 39, 23, 34, 11, 11, 24, 38, 36,  6, 34, 36,  1, 17,
       39,  5])}
1
strain_number:  0 19.105263157894736
strain_various:  0 15.11323831278952
strain_number:  1 20.526315789473685
strain_various:  1 14.247320786185997
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 22.63157894736842
strain_various:  0 12.757703663672515
strain_number:  1 20.526315789473685
strain_various:  1 7.191869832647884
fd_r_threshold:  [1.05, 626.0401674205018]
slip_r:  126.04803348410037
3
strain_number:  0 26.736842105263158
strain_various:  0 11.884718650271948
strain_number:  1 20.57894736842105
strain_various:  1 5.896616241751965
fd_r_threshold:  [1.05, 626.0401674205018, 10.839343872135709]
slip_r:  128.0059022585275
4
strain_number:  0 31.63157894736842
strain_various:  0 13.868595824325697
strain_number:  1 20.68421052631579
strain_various:  1 6.44860353368278
fd_r_thresh

glc__D_e_1:  -17.603669906993364
glc__D_e_1:  -5.720226118896177
glc__D_e_1:  -1.5166566587125212
glc__D_e_1:  -5.014796043142379
glc__D_e_1:  -16.884155101727213
glc__D_e_1:  -5.6693853620625365
glc__D_e_1:  -4.3898090410895225
glc__D_e_1:  -7.943540179288532
glc__D_e_1:  -21.36038555332616
glc__D_e_1:  -13.025652790421656
glc__D_e_1:  -6.905112025034714
glc__D_e_1:  -4.1211321743472755
glc__D_e_1:  -6.57866436458466
glc__D_e_1:  -5.80644225558069
glc__D_e_1:  -3.695653849238646
glc__D_e_1:  -2.823252319445318
glc__D_e_1:  -8.665509141608341
glc__D_e_1:  -11.75088690139864
glc__D_e_1:  -4.10989675176058
glc__D_e_1:  -2.2539101846356218
glc__D_e_1:  -8.868840602270598
glc__D_e_1:  -9.125077746530062
glc__D_e_1:  -2.119236077454462
glc__D_e_1:  -2.174827831223613
glc__D_e_1:  -4.270202921315503
glc__D_e_1:  -5.812891801210654
glc__D_e_1:  -2.0216280865324987
glc__D_e_1:  -3.552597396299096
glc__D_e_1:  -6.335558560311442
glc__D_e_1:  -6.900684548172494
glc__D_e_1:  -2.0360169506377446
g

glc__D_e_1:  -0.4563916013412084
glc__D_e_1:  -0.0407011993288815
glc__D_e_1:  -0.7666114997467219
glc__D_e_1:  -1.2808411263784385
glc__D_e_1:  -1.1301573081465932
glc__D_e_1:  -0.5260649852488775
glc__D_e_1:  -1.040294611880594
glc__D_e_1:  -0.7763125099987205
glc__D_e_1:  -0.28679875074049543
glc__D_e_1:  -1.225653912653459
glc__D_e_1:  -0.7354506521923789
strain_number:  0 2.9473684210526314
strain_various:  0 0.944439918154019
strain_number:  1 1.736842105263158
strain_various:  1 0.8486587103472157
fd_r_threshold:  [1.05, 626.0401674205018, 10.839343872135709, 8.196751720598458, 6.747437892876203, 6.947519207610328, 7.096220806601367, 8.380909425165132, 6.333064401133266, 3.674997216879588, 4.546765256160775, 5.058935463783836, 3.974320987914263, 5.223533070958875, 4.377652633538624, 4.546197404887882, 4.47428574682092, 5.294132653061224]
slip_r:  4.7831603018535045
19
glc__D_e_1:  -1.8970469352305426
glc__D_e_1:  -0.439392080411238
glc__D_e_1:  -0.8126479948546605
glc__D_e_1:  -

strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.10526315789473684
strain_various:  1 0.3068922049918579
fd_r_threshold:  [1.05, 626.0401674205018, 10.839343872135709, 8.196751720598458, 6.747437892876203, 6.947519207610328, 7.096220806601367, 8.380909425165132, 6.333064401133266, 3.674997216879588, 4.546765256160775, 5.058935463783836, 3.974320987914263, 5.223533070958875, 4.377652633538624, 4.546197404887882, 4.47428574682092, 5.294132653061224, 5.104444444444445, 3.777777777777778, 3.5, 8.86111111111111, 3.5, 3.5, 3.0, 9.0, 2.0, 0.0]
slip_r:  3.5
29
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.10526315789473684
strain_various:  1 0.3068922049918579
fd_r_threshold:  [1.05, 626.0401674205018, 10.839343872135709, 8.196751720598458, 6.747437892876203, 6.947519207610328, 7.096220806601367, 8.380909425165132, 6.333064401133266, 3.674997216879588, 4.546765256160775, 5.058935463783836, 3.9743

In [9]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)

{0: array([23, 17, 26,  8,  7, 36, 36, 25, 35,  8, 32, 24, 38, 23, 37, 30, 17,
       10,  8]), 1: array([19,  1, 32, 32,  1,  1, 25, 10, 26, 15,  8, 18, 19, 24, 28, 36, 16,
       14, 31])}
1
strain_number:  0 27.42105263157895
strain_various:  0 13.039679482151007
strain_number:  1 18.94736842105263
strain_various:  1 11.004657789990212
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.578947368421055
strain_various:  0 11.01824223957077
strain_number:  1 18.94736842105263
strain_various:  1 5.744321536691475
fd_r_threshold:  [1.05, 308.3446686508134]
slip_r:  62.508933730162674
3
strain_number:  0 38.63157894736842
strain_various:  0 11.01773941048941
strain_number:  1 18.94736842105263
strain_various:  1 4.616602580965976
fd_r_threshold:  [1.05, 308.3446686508134, 7.82562759947031]
slip_r:  63.864059250056734
4
strain_number:  0 45.94736842105263
strain_various:  0 12.296656302942154
strain_number:  1 18.94736842105263
strain_various:  1 4.925483510441827
fd_r_threshold:

glc__D_e_1:  -5.368362441226695
glc__D_e_1:  -4.032731686914509
glc__D_e_1:  -7.586462825113519
glc__D_e_1:  -12.181877794950857
glc__D_e_1:  -5.442485611283626
glc__D_e_1:  -0.5959772652213708
glc__D_e_1:  -0.15977650032470692
glc__D_e_1:  -10.738120884613817
glc__D_e_1:  -4.15317312274738
glc__D_e_1:  -2.182285679327773
glc__D_e_1:  -6.6640101010892625
glc__D_e_1:  -13.725803663280082
glc__D_e_1:  -6.163293009379547
glc__D_e_1:  -2.0334167158202856
glc__D_e_1:  -4.056178544252699
glc__D_e_1:  -6.362212904319984
glc__D_e_1:  -2.915294601585971
glc__D_e_1:  -2.7869100503178834
glc__D_e_1:  -7.2686344720793725
glc__D_e_1:  -6.098789263236925
glc__D_e_1:  -2.651870960502912
glc__D_e_1:  -2.7723900927717033
glc__D_e_1:  -6.762321995867377
glc__D_e_1:  -11.564983117480105
glc__D_e_1:  -8.992079646208733
glc__D_e_1:  -4.310992676989196
glc__D_e_1:  -3.930383665861683
glc__D_e_1:  -4.668268163315471
glc__D_e_1:  -4.719164524544805
glc__D_e_1:  -1.8102226712020864
glc__D_e_1:  -2.849399462302

glc__D_e_1:  -0.45067633920433403
glc__D_e_1:  -0.569899008464597
glc__D_e_1:  -0.5109793008809795
glc__D_e_1:  -0.35653487908018555
glc__D_e_1:  -0.1692826746186693
glc__D_e_1:  -0.9877004721983633
glc__D_e_1:  -0.8332560503975693
glc__D_e_1:  -1.781804917195434
glc__D_e_1:  -0.36201911496713945
glc__D_e_1:  -1.0219390815539553
glc__D_e_1:  -1.5361687081856719
strain_number:  0 2.9473684210526314
strain_various:  0 0.9444399181540191
strain_number:  1 1.368421052631579
strain_various:  1 0.5813347903782768
fd_r_threshold:  [1.05, 308.3446686508134, 7.82562759947031, 6.803671774921208, 8.487653809488117, 8.745047483519233, 5.40178818198332, 5.033076455338118, 4.824619371540506, 4.310232904699886, 4.592905528041143, 3.6619455253963, 5.143897062974383, 4.779812082843218, 4.783011398774385, 4.542638573948098, 3.800555555555556]
slip_r:  4.6099829348191275
18
glc__D_e_1:  -0.8788867572046242
glc__D_e_1:  -0.055768286971319525
glc__D_e_1:  -1.2885665582752903
glc__D_e_1:  -0.519341146387740

strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 308.3446686508134, 7.82562759947031, 6.803671774921208, 8.487653809488117, 8.745047483519233, 5.40178818198332, 5.033076455338118, 4.824619371540506, 4.310232904699886, 4.592905528041143, 3.6619455253963, 5.143897062974383, 4.779812082843218, 4.783011398774385, 4.542638573948098, 3.800555555555556, 6.970555555555555, 3.7569444444444446, 6.222222222222222, 5.861111111111111, 3.75, 3.5, 4.0, 1.0, 7.0, 1.0, 1.0]
slip_r:  2.8
29
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 308.3446686508134, 7.82562759947031, 6.803671774921208, 8.487653809488117, 8.745047483519233, 5.40178818198332, 5.033076455338118, 4.824619371540506, 4.310232904699886, 4.592905528041143, 3.6619455253963, 5.143897062974383, 4.779812082843218, 4.783011398774385, 4.542638573948098, 

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)

{0: array([ 0, 38, 33, 11, 24, 30, 39,  1, 17, 26, 23, 12, 17, 15,  6, 39, 11,
        4, 38]), 1: array([32, 34,  0, 16, 14, 35,  8, 32, 29, 27,  9, 30,  5, 30, 30, 28, 19,
       27, 37])}
1
strain_number:  0 23.842105263157894
strain_various:  0 15.3837342598731
strain_number:  1 23.68421052631579
strain_various:  1 11.318604301065559
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 28.210526315789473
strain_various:  0 8.127788249471983
strain_number:  1 23.94736842105263
strain_various:  1 6.700146771452307
fd_r_threshold:  [1.05, 875.4390329775937]
slip_r:  175.92780659551875
3
strain_number:  0 33.421052631578945
strain_various:  0 6.761468148527041
strain_number:  1 24.105263157894736
strain_various:  1 5.955048604177496
fd_r_threshold:  [1.05, 875.4390329775937, 7.034778082324275]
slip_r:  177.12476221198358
4
strain_number:  0 39.68421052631579
strain_various:  0 9.061501162837189
strain_number:  1 24.210526315789473
strain_various:  1 5.652935402988444
fd_r_threshol

glc__D_e_1:  -5.944575904822225
glc__D_e_1:  -2.63890862134299
glc__D_e_1:  -4.169877931109587
glc__D_e_1:  -13.014857224744741
glc__D_e_1:  -7.612813137942531
glc__D_e_1:  -3.466575942001387
glc__D_e_1:  -2.5941744122080586
glc__D_e_1:  -4.6696243470212915
glc__D_e_1:  -4.051846659818116
glc__D_e_1:  -3.041741127965185
glc__D_e_1:  -6.048087993729228
glc__D_e_1:  -7.754452999382495
glc__D_e_1:  -5.49043837171271
glc__D_e_1:  -3.2041218665810502
glc__D_e_1:  -6.210468732345094
glc__D_e_1:  -10.431658180993253
glc__D_e_1:  -7.190080661289369
glc__D_e_1:  -5.862201892978645
glc__D_e_1:  -8.432347993846022
glc__D_e_1:  -7.764132410837159
glc__D_e_1:  -5.500117783167374
glc__D_e_1:  -4.266179911756014
glc__D_e_1:  -4.869155937960132
glc__D_e_1:  -12.140810617671196
glc__D_e_1:  -10.08213677303154
glc__D_e_1:  -4.4067227415750105
glc__D_e_1:  -3.5343212117816822
glc__D_e_1:  -10.178232473364737
glc__D_e_1:  -12.440491762921733
glc__D_e_1:  -2.28128443580246
glc__D_e_1:  -1.845083670905796
g

glc__D_e_1:  -1.1849648309246008
glc__D_e_1:  -1.154781755173112
glc__D_e_1:  -2.3376854302373395
glc__D_e_1:  -0.7066093900549213
glc__D_e_1:  -1.4680172282659212
glc__D_e_1:  -0.49045433623182255
glc__D_e_1:  -0.7928448690889605
glc__D_e_1:  -1.9757485441531877
glc__D_e_1:  -0.520323734005852
glc__D_e_1:  -1.367712485969876
glc__D_e_1:  -0.33553041476279777
glc__D_e_1:  -0.8497600413945143
glc__D_e_1:  -2.000182981709198
glc__D_e_1:  -0.5803971794809035
glc__D_e_1:  -1.296678167513328
glc__D_e_1:  -2.4795818425775553
strain_number:  0 3.1052631578947367
strain_various:  0 0.9676198058342229
strain_number:  1 1.736842105263158
strain_various:  1 0.8486587103472158
fd_r_threshold:  [1.05, 875.4390329775937, 7.034778082324275, 6.482491044149134, 8.138436242882758, 8.275325156770402, 5.760789248430876, 5.088134128650866, 4.340133127881921, 4.2165672354305554, 4.0531643385360905, 7.412288061730855, 5.031090636188105, 5.1755615332908365, 4.463995860740991, 3.2125623582766445, 5.91083900226

glc__D_e_1:  -0.11813613537648648
strain_number:  0 0.10526315789473684
strain_various:  0 0.30689220499185793
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 875.4390329775937, 7.034778082324275, 6.482491044149134, 8.138436242882758, 8.275325156770402, 5.760789248430876, 5.088134128650866, 4.340133127881921, 4.2165672354305554, 4.0531643385360905, 7.412288061730855, 5.031090636188105, 5.1755615332908365, 4.463995860740991, 3.2125623582766445, 5.910839002267574, 3.1250000000000004, 6.083333333333334, 7.256944444444445, 5.861111111111111, 4.25, 4.0, 5.25, 3.0, 5.0, 2.0]
slip_r:  3.85
28
strain_number:  0 0.10526315789473684
strain_various:  0 0.30689220499185793
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 875.4390329775937, 7.034778082324275, 6.482491044149134, 8.138436242882758, 8.275325156770402, 5.760789248430876, 5.088134128650866, 4.340133127881921, 4.2165672354305554, 4.0531643385360905, 7.412288061730855, 5.031090636188105, 5.17556153

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)

{0: array([30, 26, 33, 11, 31,  5, 14, 34, 13,  4,  8, 12,  7,  9, 12, 21, 37,
       13, 35]), 1: array([ 5, 19,  2, 10, 15, 21, 25, 26, 32,  7,  7, 26, 21, 35, 14, 27, 25,
       20, 18])}
1
strain_number:  0 22.0
strain_various:  0 13.455579942126143
strain_number:  1 18.789473684210527
strain_various:  1 9.242800597269554
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 26.105263157894736
strain_various:  0 11.289933741086188
strain_number:  1 18.789473684210527
strain_various:  1 6.606049439466519
fd_r_threshold:  [1.05, 29.464961783943732]
slip_r:  6.732992356788746
3
strain_number:  0 31.05263157894737
strain_various:  0 12.513040289446788
strain_number:  1 18.789473684210527
strain_various:  1 5.652935402988443
fd_r_threshold:  [1.05, 29.464961783943732, 7.89086169393927]
slip_r:  8.101164695576601
4
strain_number:  0 36.78947368421053
strain_various:  0 13.667398579595094
strain_number:  1 18.789473684210527
strain_various:  1 5.453911656285848
fd_r_threshold:  [1.05,

glc__D_e_1:  -3.7176903447547005
glc__D_e_1:  -3.3370813336271876
glc__D_e_1:  -8.62923065826476
glc__D_e_1:  -7.702564127459996
glc__D_e_1:  -1.1971392232185871
glc__D_e_1:  -0.269145939656108
glc__D_e_1:  -8.557713346475772
glc__D_e_1:  -7.6310468156710085
strain_number:  0 15.842105263157896
strain_various:  0 7.095314672705443
strain_number:  1 7.578947368421052
strain_various:  1 4.056389235367936
fd_r_threshold:  [1.05, 29.464961783943732, 7.89086169393927, 6.7959005135679655, 7.713572176963234, 12.017398378877578, 9.731417544439495, 8.219955156435994, 4.738398800260451, 5.264660525230001, 4.815679290496413, 4.545455356045246]
slip_r:  5.5168298256936215
13
glc__D_e_1:  -6.287610794741341
glc__D_e_1:  -0.8346703467097965
glc__D_e_1:  -0.627961962434596
glc__D_e_1:  -1.667138753535378
glc__D_e_1:  -8.961178053887139
glc__D_e_1:  -1.862000665388985
glc__D_e_1:  -0.5099810289406745
glc__D_e_1:  -1.0573653013756412
glc__D_e_1:  -8.067979272838372
glc__D_e_1:  -5.135290596736076
glc__

glc__D_e_1:  -1.5197202985406966
glc__D_e_1:  -0.36595135880920093
glc__D_e_1:  -1.2408018042778857
glc__D_e_1:  -0.417683334044581
glc__D_e_1:  -0.7199777888280288
glc__D_e_1:  -1.090491519084054
glc__D_e_1:  -0.9360470972832601
glc__D_e_1:  -0.21663396915079525
glc__D_e_1:  -1.1904030794908196
glc__D_e_1:  -0.315050699023933
glc__D_e_1:  -0.3862379165968166
glc__D_e_1:  -0.9004675432285332
glc__D_e_1:  -0.48408834738504836
glc__D_e_1:  -0.8172570714872878
glc__D_e_1:  -2.000160746551515
glc__D_e_1:  -1.6548262025899156
glc__D_e_1:  -0.8500711816424025
glc__D_e_1:  -1.3643008082741193
glc__D_e_1:  -1.3236115360769538
glc__D_e_1:  -0.7710171300890436
glc__D_e_1:  -1.1501246480535345
strain_number:  0 2.789473684210526
strain_various:  0 0.8931875130777442
strain_number:  1 1.2105263157894737
strain_various:  1 0.8321783316232577
fd_r_threshold:  [1.05, 29.464961783943732, 7.89086169393927, 6.7959005135679655, 7.713572176963234, 12.017398378877578, 9.731417544439495, 8.219955156435994, 

glc__D_e_1:  -0.15097886818294604
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 29.464961783943732, 7.89086169393927, 6.7959005135679655, 7.713572176963234, 12.017398378877578, 9.731417544439495, 8.219955156435994, 4.738398800260451, 5.264660525230001, 4.815679290496413, 4.545455356045246, 3.7749930689983464, 3.7824321039228748, 5.901169111048606, 5.681641361453632, 4.215703577727388, 3.775799319727892, 5.839166666666667, 3.8333333333333335, 6.722222222222222, 5.611111111111111, 2.0, 5.5, 0.0, 7.0, 2.0, 1.0]
slip_r:  3.1
29
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 29.464961783943732, 7.89086169393927, 6.7959005135679655, 7.713572176963234, 12.017398378877578, 9.731417544439495, 8.219955156435994, 4.738398800260451, 5.264660525230001, 4.815679290496413, 4.545455356045246, 3.774993068998

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)

{0: array([ 5, 37, 24,  9,  6, 33, 25,  3,  0, 21,  2, 14, 14, 19,  0, 28, 21,
       37,  0]), 1: array([33, 28, 28,  3, 29, 24, 25,  6, 27, 29, 10, 17, 10,  2, 20, 21, 10,
       13,  3])}
1
strain_number:  0 18.42105263157895
strain_various:  0 14.8015496680644
strain_number:  1 17.842105263157894
strain_various:  1 10.116495949918091
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 21.68421052631579
strain_various:  0 8.778278404399826
strain_number:  1 17.94736842105263
strain_various:  1 7.148600777410377
fd_r_threshold:  [1.05, 602.5796131403343]
slip_r:  121.35592262806688
3
strain_number:  0 25.63157894736842
strain_various:  0 6.342924166392771
strain_number:  1 18.0
strain_various:  1 6.69642481202902
fd_r_threshold:  [1.05, 602.5796131403343, 11.52021461899519]
slip_r:  123.4499655518659
4
strain_number:  0 30.36842105263158
strain_various:  0 8.336416788725614
strain_number:  1 18.05263157894737
strain_various:  1 6.565248896636915
fd_r_threshold:  [1.05, 602.5796

glc__D_e_1:  -12.534755749803878
glc__D_e_1:  -9.807407856731711
glc__D_e_1:  -1.5706200308279326
glc__D_e_1:  -3.5933818592603455
glc__D_e_1:  -16.386562889053042
glc__D_e_1:  -8.515163391550919
glc__D_e_1:  -1.5064959628430734
glc__D_e_1:  -6.971805421936193
glc__D_e_1:  -15.084044808526205
glc__D_e_1:  -6.698415684392364
glc__D_e_1:  -3.91957544736675
glc__D_e_1:  -9.44047666022902
glc__D_e_1:  -8.793828071397112
glc__D_e_1:  -3.855117249997284
glc__D_e_1:  -1.6594559037446683
glc__D_e_1:  -5.649387806840342
glc__D_e_1:  -17.8976814325446
glc__D_e_1:  -4.522445125781598
glc__D_e_1:  -1.449288944564751
glc__D_e_1:  -8.389975959655317
glc__D_e_1:  -13.049590032890034
glc__D_e_1:  -4.9728497523577815
strain_number:  0 14.473684210526315
strain_various:  0 5.97687139645406
strain_number:  1 6.947368421052632
strain_various:  1 2.5848482437491946
fd_r_threshold:  [1.05, 602.5796131403343, 11.52021461899519, 6.707625663568732, 7.054768997704496, 8.130900688274737, 6.2890676021653995, 4.92

glc__D_e_1:  -1.1724334828820437
glc__D_e_1:  -1.0179890610812499
glc__D_e_1:  -1.017554060954435
glc__D_e_1:  -0.8631096391536409
strain_number:  0 2.789473684210526
strain_various:  0 1.5416651069344012
strain_number:  1 1.8421052631578947
strain_various:  1 0.9874559494365115
fd_r_threshold:  [1.05, 602.5796131403343, 11.52021461899519, 6.707625663568732, 7.054768997704496, 8.130900688274737, 6.2890676021653995, 4.9208964728135705, 4.36733724042613, 4.442525020800349, 5.583326456017496, 4.4532917506650955, 4.4706041002157235, 4.368886781379764, 4.039693827246388, 4.241345266793985, 4.198674099269337, 3.481354875283447]
slip_r:  4.065990969994584
19
glc__D_e_1:  -0.2269056944190213
glc__D_e_1:  -0.2868287886850114
glc__D_e_1:  -0.7538440051255557
glc__D_e_1:  -0.34276392030485536
glc__D_e_1:  -0.30140207993315826
glc__D_e_1:  -0.707956968591724
glc__D_e_1:  -1.2221865952234405
glc__D_e_1:  -0.6959603638045564
glc__D_e_1:  -0.11160094315845737
glc__D_e_1:  -1.9590581350275946
glc__D_e

glc__D_e_1:  -0.02386586473794533
strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 602.5796131403343, 11.52021461899519, 6.707625663568732, 7.054768997704496, 8.130900688274737, 6.2890676021653995, 4.9208964728135705, 4.36733724042613, 4.442525020800349, 5.583326456017496, 4.4532917506650955, 4.4706041002157235, 4.368886781379764, 4.039693827246388, 4.241345266793985, 4.198674099269337, 3.481354875283447, 7.470963718820861, 2.2222222222222223, 8.333333333333332, 1.5833333333333335, 7.0, 2.0, 2.25, 6.0, 3.0, 1.0]
slip_r:  2.85
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 602.5796131403343, 11.52021461899519, 6.707625663568732, 7.054768997704496, 8.130900688274737, 6.2890676021653995, 4.9208964728135705, 4.36733724042613, 4.442525020800349, 5.583326456017496, 4.4532917506650955, 4.470604100215

strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 602.5796131403343, 11.52021461899519, 6.707625663568732, 7.054768997704496, 8.130900688274737, 6.2890676021653995, 4.9208964728135705, 4.36733724042613, 4.442525020800349, 5.583326456017496, 4.4532917506650955, 4.4706041002157235, 4.368886781379764, 4.039693827246388, 4.241345266793985, 4.198674099269337, 3.481354875283447, 7.470963718820861, 2.2222222222222223, 8.333333333333332, 1.5833333333333335, 7.0, 2.0, 2.25, 6.0, 3.0, 1.0, 2.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0]
slip_r:  0.0
The change process of microorganism distribution is:  {0: {0: array([ 5, 37, 24,  9,  6, 33, 25,  3,  0, 21,  2, 14, 14, 19,  0, 28, 21,
       37,  0]), 1: array([33, 28, 28,  3, 29, 24, 25,  6, 27, 29, 10, 17, 10,  2, 20, 21, 10,
       13,  3])}, 1: {0: array([ 6, 44, 28, 10,  7, 39, 30,  3,  0, 25,  2, 16, 16, 22,  0, 33, 25,
       44

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)

{0: array([13, 17,  6,  1, 20,  4,  0, 34, 27, 17,  3, 11, 22, 39, 35,  7, 38,
       18, 18]), 1: array([11, 38, 10, 13, 29,  4, 35, 15,  5, 32, 33, 32, 32,  6, 35, 23, 17,
        0, 13])}
1
strain_number:  0 20.42105263157895
strain_various:  0 14.640663593939795
strain_number:  1 20.526315789473685
strain_various:  1 12.67142837066558
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 24.105263157894736
strain_various:  0 11.55260160648312
strain_number:  1 20.63157894736842
strain_various:  1 7.6585729488063485
fd_r_threshold:  [1.05, 359.5003680372378]
slip_r:  72.74007360744756
3
strain_number:  0 28.57894736842105
strain_various:  0 10.781640288611904
strain_number:  1 20.736842105263158
strain_various:  1 6.788861661521745
fd_r_threshold:  [1.05, 359.5003680372378, 9.157329751826946]
slip_r:  74.36153955781295
4
strain_number:  0 33.8421052631579
strain_various:  0 13.413516976286651
strain_number:  1 20.789473684210527
strain_various:  1 5.104991853887217
fd_r_threshol

glc__D_e_1:  -15.670179150378564
glc__D_e_1:  -10.627920268407276
glc__D_e_1:  -1.1026203724103185
glc__D_e_1:  -3.617174719508546
glc__D_e_1:  -11.53961817601555
glc__D_e_1:  -8.966714704744177
glc__D_e_1:  -4.477608177229673
glc__D_e_1:  -7.539546796762868
glc__D_e_1:  -13.939141482867742
glc__D_e_1:  -8.382652974264737
glc__D_e_1:  -2.0344818887889033
glc__D_e_1:  -4.549036235887131
glc__D_e_1:  -7.380749262671944
glc__D_e_1:  -7.122756780299691
glc__D_e_1:  -3.2971998251636343
glc__D_e_1:  -1.9330057767044915
glc__D_e_1:  -6.5288599318031375
glc__D_e_1:  -8.43133401652921
glc__D_e_1:  -3.4396084350894505
glc__D_e_1:  -1.5836218679644922
glc__D_e_1:  -4.3167068444159025
glc__D_e_1:  -5.859395724311053
glc__D_e_1:  -5.301537162868689
glc__D_e_1:  -3.4455505957437307
glc__D_e_1:  -3.7966145052312426
glc__D_e_1:  -4.001955288261371
glc__D_e_1:  -4.878213972092598
glc__D_e_1:  -3.5140199236334553
glc__D_e_1:  -5.209517837383205
glc__D_e_1:  -3.923066101747519
glc__D_e_1:  -1.63225831682

glc__D_e_1:  -1.1232362613714673
glc__D_e_1:  -0.19524297780898836
glc__D_e_1:  -0.6916186994335012
glc__D_e_1:  -1.205848326065218
glc__D_e_1:  -0.8471107662875155
glc__D_e_1:  -0.20977842933646573
glc__D_e_1:  -0.7240080559681825
glc__D_e_1:  -0.09353492562319188
glc__D_e_1:  -0.4643272505479401
glc__D_e_1:  -0.3710765720199847
glc__D_e_1:  -1.062530670312813
glc__D_e_1:  -0.6877666555242523
glc__D_e_1:  -0.8226320572781365
glc__D_e_1:  -0.6035123941576539
glc__D_e_1:  -0.8055215675483536
glc__D_e_1:  -0.09236909603875842
strain_number:  0 2.6842105263157894
strain_various:  0 0.9206766149755737
strain_number:  1 1.5789473684210527
strain_various:  1 0.6740130776245103
fd_r_threshold:  [1.05, 359.5003680372378, 9.157329751826946, 6.837698197039838, 9.50492242022279, 8.166786061848791, 5.708525841826551, 8.24562670104086, 9.372392768000154, 4.978942300403515, 4.0940633674923586, 4.64534969591591, 4.167326771829993, 3.3862752773474254, 3.265669329098401, 4.45695986536188, 3.28075538548

glc__D_e_1:  -0.09742083011751712
glc__D_e_1:  -0.023317992124582965
glc__D_e_1:  -0.033333310433846686
glc__D_e_1:  -0.05513032707379939
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 359.5003680372378, 9.157329751826946, 6.837698197039838, 9.50492242022279, 8.166786061848791, 5.708525841826551, 8.24562670104086, 9.372392768000154, 4.978942300403515, 4.0940633674923586, 4.64534969591591, 4.167326771829993, 3.3862752773474254, 3.265669329098401, 4.45695986536188, 3.2807553854875287, 5.082363945578232, 4.729166666666666, 7.222222222222222, 4.0, 6.611111111111111, 2.5, 5.5, 0.25, 5.25, 1.0, 5.0]
slip_r:  3.4
29
glc__D_e_1:  -0.08378598372451718
glc__D_e_1:  -0.07389559977118298
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 359.5003680372378, 9.157329751826946, 6.837698197039838, 9.504922420222

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)

{0: array([ 6, 39, 26, 17, 29, 26, 26,  0, 35, 17, 20, 37, 19, 34, 11, 19, 18,
       11, 24]), 1: array([35, 36, 25, 37, 27, 15,  0, 28,  1,  7, 28, 16, 13, 19,  3,  1, 28,
       10, 14])}
1
strain_number:  0 25.736842105263158
strain_various:  0 12.195090061440638
strain_number:  1 18.210526315789473
strain_various:  1 12.314214924089796
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.526315789473685
strain_various:  0 8.222999971534474
strain_number:  1 18.31578947368421
strain_various:  1 9.657572058283687
fd_r_threshold:  [1.05, 379.2959512524619]
slip_r:  76.69919025049238
3
strain_number:  0 36.21052631578947
strain_various:  0 8.179428286342068
strain_number:  1 18.473684210526315
strain_various:  1 9.051713539741533
fd_r_threshold:  [1.05, 379.2959512524619, 17.305296328357127]
slip_r:  79.95024951616381
4
strain_number:  0 43.1578947368421
strain_various:  0 11.581578648393304
strain_number:  1 18.63157894736842
strain_various:  1 8.548379509811628
fd_r_threshol

glc__D_e_1:  -6.899762910601295
glc__D_e_1:  -5.4588667531648145
glc__D_e_1:  -2.718473913719846
glc__D_e_1:  -4.741235742152259
glc__D_e_1:  -19.485566635982522
glc__D_e_1:  -7.961908052716257
glc__D_e_1:  -1.6308217938027074
glc__D_e_1:  -6.112546215564197
glc__D_e_1:  -12.400239625385307
glc__D_e_1:  -3.654825296420544
glc__D_e_1:  -2.4350103598139676
glc__D_e_1:  -5.4413572255780105
glc__D_e_1:  -10.034992777456338
glc__D_e_1:  -4.273163485823206
glc__D_e_1:  -1.991162264809657
glc__D_e_1:  -9.915434317231853
glc__D_e_1:  -14.480421799039538
glc__D_e_1:  -5.58056304827398
glc__D_e_1:  -3.994584407485203
glc__D_e_1:  -11.97444821367655
glc__D_e_1:  -11.39351324050271
glc__D_e_1:  -3.4712173817712513
glc__D_e_1:  -1.9494498547857
glc__D_e_1:  -5.447589239215558
glc__D_e_1:  -25.42932430307902
glc__D_e_1:  -6.601147548284473
strain_number:  0 14.789473684210526
strain_various:  0 8.370241443844666
strain_number:  1 6.2631578947368425
strain_various:  1 2.6125628728401806
fd_r_threshol

glc__D_e_1:  -0.41290926824913887
glc__D_e_1:  -0.37874556097979406
glc__D_e_1:  -0.5835910768785797
glc__D_e_1:  -0.15885337543590827
glc__D_e_1:  -0.004408953635114177
glc__D_e_1:  -0.8274983807384932
glc__D_e_1:  -1.179865006006577
glc__D_e_1:  -1.0254205842057829
glc__D_e_1:  -0.4049178323268723
glc__D_e_1:  -0.8535461406224232
glc__D_e_1:  -2.002131512826783
glc__D_e_1:  -1.082981635144181
glc__D_e_1:  -1.8598286606228975
glc__D_e_1:  -1.0786757487989571
glc__D_e_1:  -0.3325050038653061
glc__D_e_1:  -1.61057368099083
glc__D_e_1:  -0.5858512594837566
glc__D_e_1:  -0.4314068376829626
strain_number:  0 3.0526315789473686
strain_various:  0 1.0499967022768422
strain_number:  1 0.631578947368421
strain_various:  1 0.48237638894272
fd_r_threshold:  [1.05, 379.2959512524619, 17.305296328357127, 10.721984539895386, 10.249423924236979, 11.476015324566728, 10.784546745588571, 5.173917801330502, 3.7698627492714722, 4.600884416737547, 4.535011887831453, 7.424495575629622, 4.113529073207659, 4

glc__D_e_1:  -0.0021698514328798435
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 379.2959512524619, 17.305296328357127, 10.721984539895386, 10.249423924236979, 11.476015324566728, 10.784546745588571, 5.173917801330502, 3.7698627492714722, 4.600884416737547, 4.535011887831453, 7.424495575629622, 4.113529073207659, 4.198282449868069, 5.129273735354321, 5.871912477954145, 4.128021541950114, 6.594586167800454, 5.505555555555556, 8.194722222222222, 2.083333333333333, 2.25, 6.5, 1.0, 7.25, 0.0, 3.0, 0.0, 1.0]
slip_r:  2.25
30
glc__D_e_1:  -0.025697296976582018
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 379.2959512524619, 17.305296328357127, 10.721984539895386, 10.249423924236979, 11.476015324566728, 10.784546745588571, 5.173917801330502, 3.7698627492714722, 4.600884416737547, 4.535011887831453, 7.424495575629622, 4.1135290